In [1]:
import pandas as pd
from Bio import SeqIO
from collections import defaultdict
import statistics as st

In [18]:
# READ IN - Multiple sequence alignment
fasta="/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/RF_models_automation/RF_results/foldmason/cluster_0_1_mixed_db_out/cluster_0_1_mixed_MSTA_aa_plus_ec6098ref_2.faa"

seq_dict = {} 
for record in SeqIO.parse(fasta, "fasta"):
    seq_dict[record.id.strip()]=record.seq
seq_dict
msa_df = pd.DataFrame.from_dict(seq_dict,orient='index')
msa_df = msa_df.loc[~msa_df.index.str.contains("EC6098") ]
# msa_df.to_csv("amino_acid_switching_R/msa_df.csv")

# READ IN - features
TOP = 200 # number of features to keep
feat_df = pd.read_csv("/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/Microviridae_analysis/multiple_cluster_analysis/combined_model_phyloglm/phyloglm_intersect_giniscores_new.tsv",sep='\t')
feat_df
# Keep only features above a threshold
important = feat_df[feat_df.feature_importance_vals > 0.0001].copy()
print(important)

# READ IN - Labels
labels_df = pd.read_csv("/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/RF_models_automation/RF_results/cluster_0_1_mixed_MSTA_aa_plus_ec6_data_df.tsv", sep='\t',usecols=["protein",'ecosystem_subtype'])
print(labels_df)
labels_df = labels_df.drop_duplicates('protein').set_index("protein")
labels_df

# join the biome labels to the multiple sequence alignment

msa_df_labelled = msa_df.join(labels_df, how="left")
msa_df_labelled.to_csv("msa_df_labelled.csv")

     Unnamed: 0 feature  Estimate        SE   z.value       p.value     alpha  \
0             0   445_F -2.323584  0.722366 -3.216630  1.297059e-03  0.468187   
1             1   324_Q -2.301889  0.501361 -4.591284  4.405279e-06  0.576690   
2             2   330_K -2.588086  0.476886 -5.427056  5.729117e-08  0.521602   
3             3    19_M -2.569998  0.543479 -4.728792  2.258592e-06  0.614560   
4             4   399_L -2.499384  0.591347 -4.226596  2.372527e-05  0.545660   
..          ...     ...       ...       ...       ...           ...       ...   
572         572   265_A -1.574068  0.414116 -3.801029  1.440963e-04  0.215822   
573         573   534_N  2.654868  0.736825  3.603120  3.144197e-04  0.240998   
574         574   257_G -1.053471  0.278274 -3.785734  1.532555e-04  0.210105   
575         575   389_M -1.727666  0.485139 -3.561180  3.691921e-04  0.223072   
576         576   518_N -2.020450  0.584409 -3.457256  5.457060e-04  0.213611   

         padj  imp_order  f

In [19]:
important

,Unnamed: 0,feature,Estimate,SE,z.value,p.value,alpha,padj,imp_order,feature_importance_vals,abs_z.value,abs_Estimate,biome_predictor
0,0,445_F,-2.323584,0.722366,-3.216630,1.297059e-03,0.468187,0.009599,0,0.011341,3.216630,2.323584,lake
1,1,324_Q,-2.301889,0.501361,-4.591284,4.405279e-06,0.576690,0.000100,2,0.008046,4.591284,2.301889,lake
2,2,330_K,-2.588086,0.476886,-5.427056,5.729117e-08,0.521602,0.000002,3,0.007974,5.427056,2.588086,lake
3,3,19_M,-2.569998,0.543479,-4.728792,2.258592e-06,0.614560,0.000055,4,0.007836,4.728792,2.569998,lake
4,4,399_L,-2.499384,0.591347,-4.226596,2.372527e-05,0.545660,0.000386,5,0.007787,4.226596,2.499384,lake
...,...,...,...,...,...,...,...,...,...,...,...,...,...
572,572,265_A,-1.574068,0.414116,-3.801029,1.440963e-04,0.215822,0.001668,1597,0.000103,3.801029,1.574068,lake
573,573,534_N,2.654868,0.736825,3.603120,3.144197e-04,0.240998,0.003069,1599,0.000103,3.603120,2.654868,ocean
574,574,257_G,-1.053471,0.278274,-3.785734,1.532555e-04,0.210105,0.001744,1620,0.000101,3.785734,1.053471,lake
575,575,389_M,-1.727666,0.485139,-3.561180,3.691921e-04,0.223072,0.003511,1626,0.000101,3.561180,1.727666,lake


In [20]:
top_feats_overall = important.sort_values('feature_importance_vals', ascending=False).head(n=TOP)
top_feats_overall

,Unnamed: 0,feature,Estimate,SE,z.value,p.value,alpha,padj,imp_order,feature_importance_vals,abs_z.value,abs_Estimate,biome_predictor
0,0,445_F,-2.323584,0.722366,-3.216630,1.297059e-03,0.468187,9.599039e-03,0,0.011341,3.216630,2.323584,lake
1,1,324_Q,-2.301889,0.501361,-4.591284,4.405279e-06,0.576690,1.002775e-04,2,0.008046,4.591284,2.301889,lake
2,2,330_K,-2.588086,0.476886,-5.427056,5.729117e-08,0.521602,2.219109e-06,3,0.007974,5.427056,2.588086,lake
3,3,19_M,-2.569998,0.543479,-4.728792,2.258592e-06,0.614560,5.542130e-05,4,0.007836,4.728792,2.569998,lake
4,4,399_L,-2.499384,0.591347,-4.226596,2.372527e-05,0.545660,3.863522e-04,5,0.007787,4.226596,2.499384,lake
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,195,410_V,-1.661102,0.274371,-6.054224,1.410965e-09,0.277722,8.688289e-08,218,0.000713,6.054224,1.661102,lake
196,196,65_A,-1.493190,0.252923,-5.903739,3.553552e-09,0.373833,1.855186e-07,219,0.000713,5.903739,1.493190,lake
197,197,50_C,-2.036071,0.341249,-5.966530,2.423526e-09,0.240225,1.337954e-07,220,0.000713,5.966530,2.036071,lake
198,198,209_D,1.453732,0.444673,3.269216,1.078460e-03,0.262089,8.221975e-03,222,0.000699,3.269216,1.453732,ocean


In [21]:
# split labeled msa_df into lake and ocean ones
lake_msa = msa_df_labelled.loc[msa_df_labelled['ecosystem_subtype'] == "Lake"]
ocean_msa = msa_df_labelled.loc[msa_df_labelled['ecosystem_subtype'] == "Oceanic"]

In [22]:
# map the individual msas into dictionaries
lake_msa_map_dict = {}
for column in lake_msa:
    if column not in ['ecosystem_subtype']:
        counts = lake_msa.groupby([column])[column].count().to_dict()
        lake_msa_map_dict[column] = counts
lake_msa_map_dict

ocean_msa_map_dict = {}
for column in ocean_msa:
    if column not in ['ecosystem_subtype']:
        counts = ocean_msa.groupby([column])[column].count().to_dict()
        ocean_msa_map_dict[column] = counts
ocean_msa_map_dict

{0: {'-': 148, 'D': 1, 'L': 13, 'M': 108, 'P': 9, 'T': 7},
 1: {'-': 143,
  'A': 15,
  'E': 2,
  'F': 2,
  'G': 11,
  'H': 3,
  'K': 2,
  'L': 1,
  'M': 5,
  'N': 3,
  'P': 3,
  'Q': 3,
  'S': 78,
  'T': 2,
  'V': 13},
 2: {'-': 132,
  'A': 9,
  'D': 6,
  'E': 1,
  'F': 1,
  'G': 11,
  'I': 55,
  'K': 8,
  'L': 11,
  'M': 21,
  'N': 7,
  'P': 3,
  'Q': 2,
  'R': 4,
  'S': 14,
  'V': 1},
 3: {'-': 169,
  'E': 2,
  'F': 61,
  'G': 7,
  'H': 8,
  'I': 6,
  'L': 4,
  'M': 10,
  'P': 12,
  'R': 3,
  'T': 1,
  'Y': 3},
 4: {'-': 160,
  'A': 6,
  'G': 78,
  'H': 3,
  'I': 1,
  'M': 18,
  'N': 3,
  'P': 1,
  'Q': 2,
  'R': 2,
  'S': 3,
  'T': 9},
 5: {'-': 152,
  'A': 15,
  'D': 6,
  'E': 7,
  'G': 19,
  'K': 6,
  'L': 6,
  'M': 13,
  'N': 1,
  'P': 29,
  'Q': 4,
  'R': 15,
  'S': 3,
  'T': 4,
  'V': 2,
  'Y': 4},
 6: {'-': 141,
  'A': 15,
  'K': 8,
  'M': 5,
  'N': 29,
  'P': 2,
  'Q': 6,
  'R': 26,
  'S': 28,
  'T': 26},
 7: {'-': 133,
  'A': 3,
  'F': 1,
  'G': 56,
  'I': 14,
  'K': 2,
  'L

In [23]:
# get lake and ocean specific top features

lake_top_feats = important.loc[important['biome_predictor'] == "lake"].sort_values('feature_importance_vals', ascending=False).head(n=TOP)
ocean_top_feats = important.loc[important['biome_predictor'] == "ocean"].sort_values('feature_importance_vals', ascending=False).head(n=TOP)
ocean_top_feats.head(n=5)

# write out amino acid counts in the top X features
ocean_top_feats[['position','aa']]=ocean_top_feats['feature'].str.split("_", expand=True)
lake_top_feats[['position','aa']]=lake_top_feats['feature'].str.split("_", expand=True)
amino_acid_counts_in_feats = ocean_top_feats.groupby(['aa'])['position'].count()
amino_acid_counts_in_feats.to_csv(f"ocean_aa_counts_in{TOP}_features_NEW.csv")
amino_acid_counts_in_feats = lake_top_feats.groupby(['aa'])['position'].count()
amino_acid_counts_in_feats.to_csv(f"lake_aa_counts_in{TOP}_features_NEW.csv")

In [24]:
# get amino acids by position mapping between top_feats dataframe and the OTHER biome msa
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# ocean-important features
ocean_store_seqs = defaultdict(lambda: defaultdict(list))
ocean_store_proportions = defaultdict(lambda: defaultdict(list))
ocean_store_featcounts = defaultdict(lambda: defaultdict(list))

for index, row in ocean_top_feats.iterrows():
    feat_pos = int(row[1].split("_")[0])
    feat_AA = row[1].split("_")[1]

    total = sum(v for k, v in lake_msa_map_dict[feat_pos].items() if k not in ["X", "-"])

    for other_aa in lake_msa_map_dict[feat_pos]:
        if other_aa in ["X", "-"]:
            continue

        seqs = lake_msa_map_dict[feat_pos][other_aa]
        proportion = seqs / total

        ocean_store_seqs[feat_AA][other_aa].append(seqs)
        ocean_store_proportions[feat_AA][other_aa].append(proportion)  # was appending seqs here
        ocean_store_featcounts[feat_AA][other_aa].append(1)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# lake-important features
lake_store_seqs = defaultdict(lambda: defaultdict(list))
lake_store_proportions = defaultdict(lambda: defaultdict(list))
lake_store_featcounts = defaultdict(lambda: defaultdict(list))

for index, row in lake_top_feats.iterrows():
    feat_pos = int(row[1].split("_")[0])
    feat_AA = row[1].split("_")[1]

    total = sum(v for k, v in ocean_msa_map_dict[feat_pos].items() if k not in ["X", "-"])

    for other_aa in ocean_msa_map_dict[feat_pos]:
        if other_aa in ["X", "-"]:
            continue

        seqs = ocean_msa_map_dict[feat_pos][other_aa]
        proportion = seqs / total

        lake_store_seqs[feat_AA][other_aa].append(seqs)
        lake_store_proportions[feat_AA][other_aa].append(proportion)  # was appending seqs here
        lake_store_featcounts[feat_AA][other_aa].append(1)


/var/folders/6c/dkl_rmf913l9h4s3pys_spm80000gn/T/ipykernel_77543/1974608872.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  feat_pos = int(row[1].split("_")[0])
/var/folders/6c/dkl_rmf913l9h4s3pys_spm80000gn/T/ipykernel_77543/1974608872.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  feat_AA = row[1].split("_")[1]
/var/folders/6c/dkl_rmf913l9h4s3pys_spm80000gn/T/ipykernel_77543/1974608872.py:32: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use 

In [25]:
# either sum (for number of seqs or feat counts) or average (proportions) the lists in storage dictionaries
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# ocean-important features
ocean_store_seqs_sum = defaultdict(dict)
ocean_store_proportions_avg = defaultdict(dict)
ocean_store_featcounts_sum = defaultdict(dict)

for feat_AA,v in ocean_store_seqs.items():
    for other_AA,seqs in v.items():
        # print(seqs)
        ocean_store_seqs_sum[feat_AA][other_AA] = sum(seqs)

ocean_seqs_mat = pd.DataFrame.from_dict(ocean_store_seqs_sum)

for feat_AA,v in ocean_store_proportions.items():
    for other_AA,proportions in v.items():
        # print(seqs)
        ocean_store_proportions_avg[feat_AA][other_AA] = st.fmean(proportions)

ocean_proportions_mat = pd.DataFrame.from_dict(ocean_store_proportions_avg)

for feat_AA,v in ocean_store_featcounts.items():
    for other_AA,counts in v.items():
        # print(seqs)
        ocean_store_featcounts_sum[feat_AA][other_AA] = sum(counts)

ocean_featcounts_mat = pd.DataFrame.from_dict(ocean_store_featcounts_sum)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# lake-important features
lake_store_seqs_sum = defaultdict(dict)
lake_store_proportions_avg = defaultdict(dict)
lake_store_featcounts_sum = defaultdict(dict)

for feat_AA,v in lake_store_seqs.items():
    for other_AA,seqs in v.items():
        # print(seqs)
        lake_store_seqs_sum[feat_AA][other_AA] = sum(seqs)

lake_seqs_mat = pd.DataFrame.from_dict(lake_store_seqs_sum)

for feat_AA,v in lake_store_proportions.items():
    for other_AA,proportions in v.items():
        # print(seqs)
        lake_store_proportions_avg[feat_AA][other_AA] = st.fmean(proportions)

lake_proportions_mat = pd.DataFrame.from_dict(lake_store_proportions_avg)

for feat_AA,v in lake_store_featcounts.items():
    for other_AA,counts in v.items():
        # print(seqs)
        lake_store_featcounts_sum[feat_AA][other_AA] = sum(counts)

lake_featcounts_mat = pd.DataFrame.from_dict(lake_store_featcounts_sum)


In [26]:
#Convert matrices to long-format dataframe
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# ocean-important features
ocean_seqs_df = ocean_seqs_mat.reset_index(names="other_AA").melt(
    value_vars=["S",	"Q",	"D"	,"K",	"N",	"M",	"W",	"L"	,"F"	,"G"	,"Y"	,"H"	,"R",	"V"	,"E",	"P"	,"T",	"I"	,"C"	,"A"],
    var_name="feat_AA",
    value_name="seqs",
    id_vars=["other_AA"])

ocean_proportions_df = ocean_proportions_mat.reset_index(names="other_AA").melt(
    value_vars=["S",	"Q",	"D"	,"K",	"N",	"M",	"W",	"L"	,"F"	,"G"	,"Y"	,"H"	,"R",	"V"	,"E",	"P"	,"T",	"I"	,"C"	,"A"],
    var_name="feat_AA",
    value_name="proportion",
    id_vars=["other_AA"])

ocean_featcounts_df = ocean_featcounts_mat.reset_index(names="other_AA").melt(
    value_vars=["S",	"Q",	"D"	,"K",	"N",	"M",	"W",	"L"	,"F"	,"G"	,"Y"	,"H"	,"R",	"V"	,"E",	"P"	,"T",	"I"	,"C"	,"A"],
    var_name="feat_AA",
    value_name="feat_counts",
    id_vars=["other_AA"])

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# lake-important features
lake_seqs_df = lake_seqs_mat.reset_index(names="other_AA").melt(
    value_vars=["S",	"Q",	"D"	,"K",	"N",	"M",	"W",	"L"	,"F"	,"G"	,"Y"	,"H"	,"R",	"V"	,	"P"	,"T",	"I"	,"C"	,"A"],
    var_name="feat_AA",
    value_name="seqs",
    id_vars=["other_AA"])

lake_proportions_df = lake_proportions_mat.reset_index(names="other_AA").melt(
    value_vars=["S",	"Q",	"D"	,"K",	"N",	"M",	"W",	"L"	,"F"	,"G"	,"Y"	,"H"	,"R",	"V"	,	"P"	,"T",	"I"	,"C"	,"A"],
    var_name="feat_AA",
    value_name="proportion",
    id_vars=["other_AA"])

lake_featcounts_df = lake_featcounts_mat.reset_index(names="other_AA").melt(
    value_vars=["S",	"Q",	"D"	,"K",	"N",	"M",	"W",	"L"	,"F"	,"G"	,"Y"	,"H"	,"R",	"V"	,	"P"	,"T",	"I"	,"C"	,"A"],
    var_name="feat_AA",
    value_name="feat_counts",
    id_vars=["other_AA"])


In [27]:
print(ocean_seqs_df.shape)
print(ocean_proportions_df.shape)
print(ocean_featcounts_df.shape)

print(lake_seqs_df.shape)
print(lake_proportions_df.shape)
print(lake_featcounts_df.shape)

(400, 3)
(400, 3)
(400, 3)
(380, 3)
(380, 3)
(380, 3)


In [28]:
# merge all the dataframes to make one dataframe for heatmap
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# ocean-important features
merged_df_1 = ocean_seqs_df.merge(ocean_proportions_df,
    how="left",
    left_on=["other_AA","feat_AA"],
    right_on=["other_AA","feat_AA"]
)

ocean_merged_df_2 = merged_df_1.merge(ocean_featcounts_df,
                                how="left",
                                left_on=["other_AA","feat_AA"],
                                right_on=["other_AA","feat_AA"])
print(ocean_merged_df_2)
ocean_merged_df_2.to_csv(f"ocean_switch_df_for_heatmap_{TOP}_NEW.tsv",index=False, sep='\t')

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# lake-important features
merged_df_1 = lake_seqs_df.merge(lake_proportions_df,
    how="left",
    left_on=["other_AA","feat_AA"],
    right_on=["other_AA","feat_AA"]
)

lake_merged_df_2 = merged_df_1.merge(lake_featcounts_df,
                                how="left",
                                left_on=["other_AA","feat_AA"],
                                right_on=["other_AA","feat_AA"])
print(lake_merged_df_2)
lake_merged_df_2.to_csv(f"lake_switch_df_for_heatmap_{TOP}_NEW.tsv",index=False, sep='\t')



    other_AA feat_AA    seqs  proportion  feat_counts
0          A       S  2106.0    0.180745         15.0
1          F       S    54.0    0.021674          6.0
2          G       S  3794.0    0.330579         14.0
3          H       S   260.0    0.080753          8.0
4          I       S    35.0    0.006299          7.0
..       ...     ...     ...         ...          ...
395        E       A    32.0    0.010683          4.0
396        L       A    56.0    0.010270          7.0
397        D       A    97.0    0.032464          4.0
398        C       A   202.0    0.062520          4.0
399        W       A     3.0    0.003717          1.0

[400 rows x 5 columns]
    other_AA feat_AA   seqs  proportion  feat_counts
0          D       S  345.0    0.179665          8.0
1          F       S   57.0    0.034793          6.0
2          G       S  156.0    0.061135         10.0
3          H       S  258.0    0.256661          4.0
4          I       S   32.0    0.023784          5.0
..       .